# TraceFusion — Merging Multi-Rank Traces for Visual Analysis

In distributed deep learning, diagnosing issues like straggling ranks, load imbalance, or communication bottlenecks requires a **global view** of events across all ranks. TraceFusion merges trace files from multiple ranks into a single file for seamless rendering in **Perfetto**.

## Setup & Imports

In [1]:
import os
import json
from TraceLens import TraceFuse

## Define Profile Paths

We have a 4-GPU DDP ResNet training trace with one file per rank.

In [ ]:
profiles_root = "HPCTrainingExamples/MLExamples/TraceLens"
output_dir = os.path.join(profiles_root, "fused_traces")
os.makedirs(output_dir, exist_ok=True)

world_size = 4
list_profile_files = [
    os.path.join(profiles_root, f"traces/trace_0_20_{i}.json") for i in range(world_size)
]

# Verify all files exist
for f in list_profile_files:
    assert os.path.exists(f), f"Missing: {f}"
    print(f"Exists {os.path.basename(f)} ({os.path.getsize(f) / 1e6:.1f} MB)")

Exists trace_0_20_0.json (14.0 MB)
Exists trace_0_20_1.json (14.0 MB)
Exists trace_0_20_2.json (14.0 MB)
Exists trace_0_20_3.json (14.0 MB)


## 1. Basic Merge — All Events, All Ranks

The simplest usage: merge all events from all ranks into a single trace file. By default, Python function category events (`python_function`) are skipped to save memory.

In [ ]:
# Initialize TraceFusion
fuser = TraceFuse(list_profile_files)

# Merge and save all events
output_all = os.path.join(output_dir, "merged_all_events.json")
fuser.merge_and_save(output_all)

print(f"Merged trace saved to: {output_all}")

The merged trace can be loaded in [Perfetto UI](https://ui.perfetto.dev/) to view all ranks side by side:

![TraceFusion merged trace in Perfetto](images/trace_fusion.png "Merged trace — all events, all ranks")

## 2. Filter: NCCL Communication Events Only

To diagnose inter-rank communication, filter to only NCCL kernel and annotation events. This creates a much smaller file focused on collective operations (`all_reduce`, `broadcast`).

In [ ]:
def filter_nccl_kernels(event):
    """Keep only NCCL kernel and gpu_user_annotation events."""
    cat = event.get('cat', '')
    name = event.get('name', '').lower()
    return cat in ['kernel', 'gpu_user_annotation'] and 'nccl' in name

fuser = TraceFuse(list_profile_files)
output_nccl = os.path.join(output_dir, "merged_nccl_only.json")
fuser.merge_and_save(output_nccl, filter_fn=filter_nccl_kernels)

print(f"NCCL-only trace saved to: {output_nccl}")


With the NCCL filter, the Perfetto view shows only the collective communication events across ranks — making it easy to spot stragglers and timing mismatches:

![TraceFusion NCCL-only trace in Perfetto](images/trace_fusion_nccl.png "Merged trace — NCCL events only")